In [4]:
import pandas as pd
import numpy as np
import re

import os
import requests
import urllib
import ast
import json
import translators as ts
from time import sleep

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

## MOMA

### Artworks Dataset

In [2]:
moma_works = pd.read_csv('https://media.githubusercontent.com/media/MuseumofModernArt/collection/refs/heads/main/Artworks.csv')
moma_works.head()

,Title,Artist,ConstituentID,ArtistBio,Nationality,BeginDate,EndDate,Gender,Date,Medium,...,OnView,Circumference (cm),Depth (cm),Diameter (cm),Height (cm),Length (cm),Weight (kg),Width (cm),Seat Height (cm),Duration (sec.)
0,"Ferdinandsbrücke Project, Vienna, Austria (Ele...",Otto Wagner,6210,"(Austrian, 1841–1918)",(Austrian),(1841),(1918),(male),1896,Ink and cut-and-pasted painted pages on paper,...,NaN,NaN,NaN,NaN,48.6000,NaN,NaN,168.9000,NaN,NaN
1,"City of Music, National Superior Conservatory ...",Christian de Portzamparc,7470,"(French, born 1944)",(French),(1944),(0),(male),1987,Paint and colored pencil on print,...,NaN,NaN,NaN,NaN,40.6401,NaN,NaN,29.8451,NaN,NaN
2,"Villa project, outside Vienna, Austria (Elevat...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(male),1903,"Graphite, pen, color pencil, ink, and gouache ...",...,NaN,NaN,NaN,NaN,34.3000,NaN,NaN,31.8000,NaN,NaN
3,"The Manhattan Transcripts Project, New York, N...",Bernard Tschumi,7056,"(French and Swiss, born Switzerland 1944)",(),(1944),(0),(male),1980,Photographic reproduction with colored synthet...,...,NaN,NaN,NaN,NaN,50.8000,NaN,NaN,50.8000,NaN,NaN
4,"Villa project, outside Vienna, Austria (Exteri...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(male),1903,"Graphite, color pencil, ink, and gouache on tr...",...,NaN,NaN,NaN,NaN,38.4000,NaN,NaN,19.1000,NaN,NaN


In [3]:
moma_works.columns

Index(['Title', 'Artist', 'ConstituentID', 'ArtistBio', 'Nationality',
       'BeginDate', 'EndDate', 'Gender', 'Date', 'Medium', 'Dimensions',
       'CreditLine', 'AccessionNumber', 'Classification', 'Department',
       'DateAcquired', 'Cataloged', 'ObjectID', 'URL', 'ImageURL', 'OnView',
       'Circumference (cm)', 'Depth (cm)', 'Diameter (cm)', 'Height (cm)',
       'Length (cm)', 'Weight (kg)', 'Width (cm)', 'Seat Height (cm)',
       'Duration (sec.)'],
      dtype='object')

1) Excluding entries without supporting images for the conseqent visual analysis.
2) Choosing a subset of works by Finnish authors.

In [243]:
moma_works_w_imgs = moma_works.query('ImageURL.notna()')
fin_works = moma_works_w_imgs[moma_works_w_imgs.Nationality.str.contains('Finnish', na=False)]
fin_works.shape

(175, 30)

In [5]:
fin_works.isna().sum()

Title                   0
Artist                  0
ConstituentID           0
ArtistBio               0
Nationality             0
BeginDate               0
EndDate                 0
Gender                  0
Date                    0
Medium                  0
Dimensions              2
CreditLine              0
AccessionNumber         0
Classification          0
Department              0
DateAcquired            0
Cataloged               0
ObjectID                0
URL                     0
ImageURL                0
OnView                175
Circumference (cm)    175
Depth (cm)            123
Diameter (cm)          98
Height (cm)            12
Length (cm)           167
Weight (kg)           175
Width (cm)             87
Seat Height (cm)      175
Duration (sec.)       174
dtype: int64

Choosing a subset of potentially relevant columns – coincidentally, without missing values.

In [244]:
fin_works = fin_works[['Title', 'Artist', 'ConstituentID', 'Date', 'Medium', 'CreditLine', 'AccessionNumber', 
       'Classification', 'Department', 'DateAcquired', 'Cataloged', 'ObjectID', 'URL', 'ImageURL']]

Time range we are working with is 1907-2015.

In [20]:
fin_works.Date.sort_values().unique()

array(['1907', '1927-35', '1929', '1930', '1930s', '1930–1932',
       '1931-1932', '1931-32', '1931–1932', '1932', '1932-33',
       '1932–1933', '1936', '1936-37', '1936–1937', '1936–1956', '1944',
       '1946', '1946-47', '1947', '1948', '1950', '1950s', '1951', '1952',
       '1953', '1953-57', '1954', '1955', '1955-58', '1956', '1956-1957',
       '1957', '1958', '1958-87', '1959', '1960', '1960s', '1961', '1962',
       '1963', '1964', '1967', '1968', '1969', '1970', '1977', '1978',
       '1979', '1987', '1987-91', '1989', '1990-92', '1990-94', '1996',
       '1997', '1999', '1999-2000', '2000-2001', '2000/2003', '2001',
       '2001-03', '2001-2002', '2002', '2002-2003', '2003', '2005',
       '2009', '2012', '2015', 'Designed 1929 (this example ca. 1936)',
       'Designed 1960-1967, this example 1976', 'Late 1930s', 'Unknown',
       'c. 1930', 'c. 1932', 'c. 1949-62', 'c. 1949-63', 'c. 1950',
       'c. 1950-62', 'c. 1953', 'c. 1955', 'c. 1956', 'c. 1969',
       'c. 1970',

Years need to be brought up to a unified format for analysis

In [271]:
def approx_date(d):
  d = d.replace("c. ", "")
  if d.isnumeric():
    return int(d)
  if re.match(r"\d{4}s", d):
    return int(d[:4])
  if d == 'Late 1930s':
    return 1930
  if d == '2000/2003':
    return 2000
  if d == 'Designed 1929 (this example ca. 1936)':
    return 1929
  if d == 'Designed 1960-1967, this example 1976':
    return 1967
  if d == 'Designed 1960-1967, this example 1976':
    return 1967
  if '–' in d:
    return int(d.split('–')[0])
  if '-' in d:
    return int(d.split('-')[0])
  return None


fin_works['ApproxDate'] = fin_works.Date.apply(approx_date)

Collecting the artwork images.

In [ ]:
fin_art_map = {}

# To bypass the 403 Forbidden error, we mimic a browser by adding a User-Agent header.
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}

def get_artwork_image_moma(entry):
  img_url = entry.ImageURL
  work_id = entry.ObjectID

  artist_ids = entry.ConstituentID.split(', ')
  for a_id in artist_ids:
    if a_id not in fin_art_map:
      fin_art_map[a_id] = set()
    fin_art_map[a_id].add(work_id)

  req = urllib.request.Request(img_url, headers=headers)
  if not os.path.exists('moma_imgs'):
    os.mkdir('moma_imgs')
  with urllib.request.urlopen(req) as resp:
      with open(f'moma_imgs/{work_id}.jpg', 'wb') as f:
          f.write(resp.read())


fin_works.apply(get_artwork_image_moma, axis=1)

In [ ]:
# Saving the images (archiving for downloading from Colab)
!zip -r /content/moma_imgs.zip /content/moma_imgs

### Material Design Subset

Because the different forms of artistic expression are hard to compare, we are limiting the dataset to one prevalent category – Design.

In [11]:
fin_works.Classification.value_counts()

Design          140
Drawing          25
Architecture      8
Periodical        1
Video             1
Name: Classification, dtype: int64

In [273]:
moma_finnish_design = fin_works.query('Classification == "Design"')
moma_finnish_design.Artist.value_counts()

Kaj Franck                                          31
Alvar Aalto                                         25
Tapio Wirkkala                                      17
Timo Sarpaneva                                      13
Saara Hopea                                          9
Caroline Slotte                                      5
Nanny Still                                          3
Göran Hongell                                        3
Antti Nurmesniemi                                    3
Eero Aarnio                                          2
Yki Nummi                                            2
Bertel Gardberg                                      2
Arabia, Wärtsilä Ab, Helsinki, Finland               2
Friedl Holzer-Kjellberg                              1
Akseli Gallen-Kallela                                1
Eliel Saarinen                                       1
Taneli Armanto                                       1
Maria Jauhiainen                                     1
Alvar Aalt

In [71]:
fin_design_map = {}
for row in moma_finnish_design.itertuples():
  work_id = row.ObjectID
  artist_ids = map(int, row.ConstituentID.split(', '))
  for a_id in artist_ids:
    if a_id not in fin_design_map:
      fin_design_map[a_id] = set()
    fin_design_map[a_id].add(work_id)

### Artists Dataset

In [6]:
moma_artists = pd.read_csv('https://media.githubusercontent.com/media/MuseumofModernArt/collection/refs/heads/main/Artists.csv')
moma_artists.head()

,ConstituentID,DisplayName,ArtistBio,Nationality,Gender,BeginDate,EndDate,Wiki QID,ULAN
0,1,Robert Arneson,"American, 1930–1992",American,male,1930,1992,NaN,NaN
1,2,Doroteo Arnaiz,"Spanish, born 1936",Spanish,male,1936,0,NaN,NaN
2,3,Bill Arnold,"American, born 1941",American,male,1941,0,NaN,NaN
3,4,Charles Arnoldi,"American, born 1946",American,male,1946,0,Q1063584,500027998.0
4,5,Per Arnoldi,"Danish, born 1941",Danish,male,1941,0,NaN,NaN


Leaving only designers, whose works are present in the dataset above.

In [72]:
finnish_designers = moma_artists[moma_artists.ConstituentID.isin(fin_design_map.keys())]
finnish_designers

,ConstituentID,DisplayName,ArtistBio,Nationality,Gender,BeginDate,EndDate,Wiki QID,ULAN
28,34,Alvar Aalto,"Finnish, 1898–1976",Finnish,male,1898,1976,Q82840,500002617.0
29,35,Aino Aalto,"Finnish, 1894–1949",Finnish,female,1894,1949,Q273511,500024110.0
30,36,Eero Aarnio,"Finnish, born 1932",Finnish,male,1932,0,Q707025,500270610.0
234,271,Olof Backstrom,"Finnish, 1922–1998",Finnish,male,1922,1998,NaN,NaN
1732,1968,Kaj Franck,"Finnish, 1911–1989",Finnish,male,1911,1989,Q909809,500103402.0
1815,2059,Akseli Gallen-Kallela,"Finnish, 1865–1931",Finnish,male,1865,1931,Q170068,500015305.0
2405,2715,Friedl Holzer-Kjellberg,"Finnish, born Austria. 1905–1993",Finnish,male,1905,1993,NaN,NaN
2409,2720,Göran Hongell,"Finnish, 1902–1973",Finnish,male,1902,1973,NaN,NaN
2413,2724,Saara Hopea,"Finnish, 1925–1984",Finnish,female,1925,1984,NaN,NaN
2618,2953,Dora Jung,"Finnish, 1906–1980",Finnish,female,1906,1980,Q11856286,NaN


Removing companies (via a stopword list) and leaving only individual artists

In [ ]:
finnish_designers_upd = finnish_designers[~finnish_designers.DisplayName.str.contains("Karhula|Arabia|OFFECCT")]
finnish_designers_upd.to_csv("moma_designers.csv")
designer_list = finnish_designers_upd.DisplayName.tolist()

Updating the artworks dataset to include only artists above

In [ ]:
filtering_rule = lambda x: any([int(i) in finnish_designers_upd.ConstituentID.unique() for i in x.split(', ')])
moma_finnish_design_upd = moma_finnish_design[moma_finnish_design.ConstituentID.apply(filtering_rule)]

Editing authorship for artist + company collaborations

In [275]:
moma_finnish_design_upd.loc[moma_finnish_design_upd.ObjectID == 82135, ["Artist", "ConstituentID"]] = ["Teppo Asikainen", 22564]
moma_finnish_design_upd.loc[moma_finnish_design_upd.ObjectID == 89289, ["Artist", "ConstituentID"]] = ["Alvar Aalto", 34]

Resulting size of the artworks dataset

In [276]:
moma_finnish_design_upd['YearAcquired'] = moma_finnish_design_upd.DateAcquired.apply(lambda x: int(x.split('-')[0]))
moma_finnish_design_upd.drop(['Classification', 'Department', 'Cataloged', 'URL', 'ImageURL'], axis=1).to_csv("moma_designs.csv")
moma_finnish_design.shape, moma_finnish_design_upd.shape

((140, 15), (137, 16))

In [ ]:
fin_design_map_upd = {key: list(val) for key, val in fin_design_map.items() if key in finnish_designers_upd.ConstituentID.unique()}
with open('fin_design_map.json', 'w') as f:
  json.dump(fin_design_map_upd, f)

Resulting number of the artists

In [257]:
len(fin_design_map), len(fin_design_map_upd)

(39, 35)

## FINNA

Query parameters:
- Authors from the **designer_list**
- Format = **Physical object**
- Record has an image of an object (Media Type = **Image**)

In [ ]:
finna_data_list = []

def get_finna_artworks(artist):
  page_n = 1
  while True:
    api_url = f'https://api.finna.fi/api/v1/search?lookfor0%5B%5D={'+'.join(artist.split())}&type0%5B%5D=Author&sort=main_date_str+asc&limit=100&filter%5B%5D=~format%3A"0%2FPhysicalObject%2F"&filter%5B%5D=~media_type_str_mv%3A"0%2Fimage%2F"{"&field%5B%5D=".join(["", "id", "images", "nonPresenterAuthors", "subjects", "title", "year"])}&page={page_n}'
    try:
        response = requests.get(api_url)
        response.raise_for_status()
        data = response.json()
        if "records" not in data or data["resultCount"] < (page_n - 1) * 100:
          break
        page_n += 1
        for record in data['records']:
          record['IndividualAuthor'] = artist
          finna_data_list.append(record)

    except Exception as e:
      print(f"Author: {artist}")
      print(f"Page: {page_n}")
      print(f"Error: {e}")
      break


for designer in designer_list:
  get_finna_artworks(designer)

finna_artworks = pd.DataFrame(finna_data_list)
finna_artworks.shape

(1664, 7)

In [ ]:
finna_artworks

,id,images,nonPresenterAuthors,subjects,title,year,IndividualAuthor
0,lappeenrannanmuseot.0188C337-B8BA-4027-B85E-AE...,[/Cover/Show?source=Solr&id=lappeenrannanmuseo...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[[huonekalut], [naulakot], [seinähyllyt], [Luu...",naulakko; seinänaulakko,1930,Alvar Aalto
1,lappeenrannanmuseot.3db2044e-1b19-4514-903d-5d...,[/Cover/Show?source=Solr&id=lappeenrannanmuseo...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[[tuolit], [sairaalat], [tuberkuloosiparantolat]]",tuoli,1930,Alvar Aalto
2,lappeenrannanmuseot.48FD4DBA-A6F7-4184-AB74-77...,[/Cover/Show?source=Solr&id=lappeenrannanmuseo...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[[muotoilu], [huonekalut], [päiväkodit], [last...",selkänojallinen pikkutuoli,1930,Alvar Aalto
3,lappeenrannanmuseot.6219dd3c-87f5-470e-bd59-04...,[/Cover/Show?source=Solr&id=lappeenrannanmuseo...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[[tuolit], [sairaalat], [tuberkuloosiparantolat]]",tuoli,1930,Alvar Aalto
4,lappeenrannanmuseot.67B47DF2-AA24-4415-A480-30...,[/Cover/Show?source=Solr&id=lappeenrannanmuseo...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[[naulakot], [huonekalut], [seinähyllyt]]",naulakko; seinänaulakko,1930,Alvar Aalto
...,...,...,...,...,...,...,...
1659,postimuseo.FE49BC51-EB37-4BC7-B019-820136B04F71,[/Cover/Show?source=Solr&id=postimuseo.FE49BC5...,"[{'name': 'Saarinen, Eliel', 'role': 'tekijä',...","[[luonnokset], [postimerkkien valmistus], [vaa...","vedos; postimerkin mallivedos, ns. Bernin arkk...",NaN,Eliel Saarinen
1660,srm.166901944222200,[/Cover/Show?source=Solr&id=srm.16690194422220...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[[huonekalut], [istuimet], [toimistokalusteet]...",istuin; tuoli,NaN,Eliel Saarinen
1661,srm.166903323327000,[/Cover/Show?source=Solr&id=srm.16690332332700...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[[huonekalut], [istuimet], [tuolit]]",istuin; tuoli,NaN,Eliel Saarinen
1662,srm.166903329663700,[/Cover/Show?source=Solr&id=srm.16690332966370...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[[asemaravintolat], [huonekalut], [istuimet], ...",asemaravintolan tuoli; istuin; tuoli,NaN,Eliel Saarinen


In [ ]:
finna_artworks.to_csv("finna_designs_raw.csv")

In [126]:
finna_designs = pd.read_csv("finna_designs_raw.csv", index_col=0)
finna_designs.isna().sum()

id                       0
images                   0
nonPresenterAuthors      0
subjects                 0
title                    0
year                   314
IndividualAuthor         0
dtype: int64

314 out of 1664 records arew missing a **year** parameter. Filling them with a median value for a corresponding artist.

In [128]:
new_years = []

for entry in finna_designs.itertuples():
  author = entry.IndividualAuthor
  subset = finna_designs[finna_designs.IndividualAuthor == author].year
  if entry.year.is_integer():
    new_years.append(entry.year)
  else:
    new_years.append(round(subset.median()))

finna_designs.year = new_years

Downloading images

In [ ]:
def get_artwork_image_finna(entry):
  img_url = f"https://www.finna.fi/Cover/Show?source=Solr&id={entry.id}&index=0&size=large"
  try:
    req = urllib.request.Request(img_url, headers=headers)
    if not os.path.exists('finna_imgs'):
      os.mkdir('finna_imgs')
    with urllib.request.urlopen(req) as resp:
        with open(f'finna_imgs/{entry.id}.jpg', 'wb') as f:
            f.write(resp.read())
  except Exception as e:
    print(f"Index: {entry.name}")
    print(f"Error: {e}")
    print()

finna_designs.apply(get_artwork_image_finna, axis=1)

In [ ]:
!zip -r /content/finna_imgs.zip /content/finna_imgs

There are 2 artworks with 2 authors from the list

In [170]:
finna_designs.id.value_counts()

ilomantsi.knp-295277                                        2
alvaraalto.aam-118695                                       2
lappeenrannanmuseot.0188C337-B8BA-4027-B85E-AE78D062092A    1
kymenlaaksonmuseot.KarhE1373:11                             1
museovirasto.2FF7482E1407A5C810BB6643025052D5               1
                                                           ..
kymenlaaksonmuseot.KarhE959:1                               1
kymenlaaksonmuseot.KarhE957:3                               1
kymenlaaksonmuseot.KarhE957:2                               1
kymenlaaksonmuseot.KarhE941:4                               1
srm.166903331144400                                         1
Name: id, Length: 1662, dtype: int64

In [171]:
finna_designs.query("id.isin(['ilomantsi.knp-295277', 'alvaraalto.aam-118695'])")

,id,images,nonPresenterAuthors,subjects,title,year,IndividualAuthor
45,alvaraalto.aam-118695,['/Cover/Show?source=Solr&id=alvaraalto.aam-11...,"[{'name': 'Alvar Aalto', 'role': 'Suunnittelij...",[],Riihimäen kukka,NaN,Alvar Aalto
105,alvaraalto.aam-118695,['/Cover/Show?source=Solr&id=alvaraalto.aam-11...,"[{'name': 'Alvar Aalto', 'role': 'Suunnittelij...",[],Riihimäen kukka,NaN,Aino Aalto
186,ilomantsi.knp-295277,['/Cover/Show?source=Solr&id=ilomantsi.knp-295...,"[{'name': 'Still, Nanny', 'role': 'Suunnitteli...","[['snapsilasit'], ['astiat'], ['juomalasit'], ...",Karahvisetti; Karahvi; Snapsilasi,1950.0,Kaj Franck
957,ilomantsi.knp-295277,['/Cover/Show?source=Solr&id=ilomantsi.knp-295...,"[{'name': 'Still, Nanny', 'role': 'Suunnitteli...","[['snapsilasit'], ['astiat'], ['juomalasit'], ...",Karahvisetti; Karahvi; Snapsilasi,1950.0,Nanny Still


I decided to leave one less represented artist as a main author to diversify the authorship attributions.

In [ ]:
finna_designs.loc[finna_designs.id == "alvaraalto.aam-118695", "IndividualAuthor"] = "Aino Aalto"
finna_designs.loc[finna_designs.id == "ilomantsi.knp-295277", "IndividualAuthor"] = "Nanny Still"
finna_designs.drop_duplicates('id', inplace=True)
finna_designs.to_csv("finna_designs.csv")

## Data Processing

In [217]:
moma = pd.read_csv("moma_designs.csv", index_col=0)
finna = pd.read_csv("finna_designs.csv", index_col=0)
artists = pd.read_csv("moma_designers.csv", index_col=0)

In [222]:
moma.sort_values('DateAcquired').head(10)

,Title,Artist,ConstituentID,Date,Medium,CreditLine,AccessionNumber,DateAcquired,ObjectID,ApproxDate,YearAcquired
3492,Stacking Armchair (model 403),Alvar Aalto,34,1931–1932,Lacqured birch and enamelled birch plywood,Purchase,833.1942,1942-01-01,4460,1931.0,1942
3434,Paimio Lounge Chair (model 41),Alvar Aalto,34,1931-1932,Laminated birch and lacquered molded plywood,"Edgar Kaufmann, Jr. Fund",711.1943,1943-11-04,4385,1931.0,1943
3436,Savoy Vase,Alvar Aalto,34,1936–1937,Mold-blown glass,"Gift of Artek-Pascoe, Inc.",712.1943,1943-11-04,4387,1936.0,1943
1264,Stacking Stools (model 60),Alvar Aalto,34,1932–1933,Laminated birch and lacquered plywood,"Gift of Artek-Pascoe, Inc.",56.1946.1-3,1946-05-07,1882,1932.0,1946
1263,Stacking Stools (model 60),Alvar Aalto,34,1932-33,Birch,"Gift of Artek-Pascoe, Inc.",56.1946.1,1946-05-07,1881,1932.0,1946
1255,Child's Chair (model 103),"Alvar Aalto, Aino Aalto","34, 35",Designed 1929 (this example ca. 1936),Laminated birch and lacquered molded plywood,"Gift of Artek-Pascoe, Inc.",55.1946,1946-05-07,1872,1929.0,1946
1250,Dish,Alvar Aalto,34,Late 1930s,Glass,Gift of Mrs. Susanne Wasson-Tucker,54.1946,1946-05-07,1866,1930.0,1946
1265,Stacking Stools (model 60),Alvar Aalto,34,1932-33,Birch,"Gift of Artek-Pascoe, Inc.",56.1946.2,1946-05-07,1883,1932.0,1946
1469,Bowl,Göran Hongell,2720,1930s,Crystal,Gift of Finland Ceramics & Glass Corp.,109.1948,1948-03-17,2150,1930.0,1948
2053,Munankuori Bowl (model 3303),Gunnel Nyman,4352,1947,Blown glass,Gift of Finland Ceramics and Glass Corp.,228.1950,1950-06-08,2843,1947.0,1950


### MOMA

For artworks that do not have a date attribution and other artworks by the same author to refer to, I roughly estimate it as a middle of an artist's life.

In [4]:
moma[moma.ApproxDate.isna()]

,Title,Artist,ConstituentID,Date,Medium,CreditLine,AccessionNumber,DateAcquired,ObjectID,ApproxDate,YearAcquired
132689,Untitled,Eliel Saarinen,64766,n.d.,Pencil on paper,Gift of the Gilbert B. and Lila Silverman Inst...,2130.2018,2018-11-05,289373,NaN,2018


In [7]:
artist_info = moma_artists[moma_artists.ConstituentID == 64766]
moma.loc[moma.ObjectID == 289373, 'ApproxDate'] = (artist_info.BeginDate + round((artist_info.EndDate - artist_info.BeginDate) / 2)).values[0]

Assigning artist a position number in alphabetical order for visualisation

In [ ]:
artist_order = sorted(list(artists.DisplayName), key=lambda x: x.split()[1] + ' ' + x.split()[0])
artist_order_map = {name: i + 1 for i, name in enumerate(artist_order)}

moma['ArtistOrder'] = moma.Artist.apply(lambda x: artist_order_map[x] if ',' not in x else artist_order_map[x.split(',')[0]]) # assuming that the author first in the list is a primary one
finna['ArtistOrder'] = finna.IndividualAuthor.apply(lambda x: artist_order_map[x] if ',' not in x else artist_order_map[x.split(',')[0]])

Unifying the artwork medium (material). Given that some works are composite, automated grouping via keyword is not applicable here.

Therefore, I manually compiled a dictionary, that groups mediums based on the primary material.

In [318]:
moma.Medium.unique()

array(['Pressed glass', 'Silver and teak', 'Wood and stainless steel',
       'Glazed earthenware', 'Blown glass',
       'Hand-cast glass, cord, and electrical switch', 'Glass',
       'Laminated birch and lacquered molded plywood', 'Birch',
       'Laminated birch and lacquered plywood', 'Cast iron and teak',
       'Satin damask', 'Crystal', 'Paper twine', 'Turn mold-blown glass',
       'Laminated birch and teak', 'Wood and cotton webbing',
       'Fiberglass',
       'Chrome-plated steel frame, wool upholstery, and painted wood arms',
       'Birch and leather',
       'Linoleum top, natural birch frame, and lacquered wheels with rubber tread',
       'Glazed porcelain', 'Plywood', 'Glazed earthenware (hard faience)',
       'Laminated Palisander and teak wood', 'Porcelain',
       'Steam-blown glass', 'Steel and glass', 'Acrylic', 'Teak',
       'Solid and laminated birch and plywood',
       'Chrome-plated tubular steel and molded plywood',
       'Solid and laminated birch', 'L

In [142]:
material_map = {'Wood': [
                        'Birch', 
                        'Teak',
                        'Plywood', 
                        'Laminated birch and teak', 
                        'Laminated Palisander and teak wood',
                        'Solid and laminated birch and plywood',
                        'Laminated birch and upholstery',
                        'Birch and upholstery',
                        'Birch and leather',
                        'Wood and cotton webbing',
                        'Laminated birch, linoleum, and rubber',
                        'Solid and laminated wood and rattan',
                        'Solid and laminated birch', 
                        'Laminated birch and rattan',
                        'Laminated teak and palisander wood',
                        'Laminated birch and lacquered molded plywood',
                        'Laminated birch and lacquered plywood', 
                        'Lacqured birch and enamelled birch plywood',
                        'Chrome-plated tubular steel and molded plywood',
                        'Linoleum top, natural birch frame, and lacquered wheels with rubber tread'
                        ],
                'Glass': [
                        'Pressed glass', 
                        'Blown glass',
                        'Hand-cast glass, cord, and electrical switch',
                        'Crystal',
                        'Turn mold-blown glass',
                        'Fiberglass',
                        'Glass and wood',
                        'Steel and glass',
                        'Molded glass',
                        'Steam-blown glass',
                        'Mold-blown glass',
                        'Hand-shaped, sand-blasted, and acid-polished glass',
                        'Clear and white opal glass'
                        ],
                'Ceramics': [
                        'Glazed earthenware',
                        'Glazed porcelain',
                        'Glazed earthenware (hard faience)', 
                        'Porcelain',
                        'Earthenware',
                        'Reworked ceramic'
                        ],
                'Metal': [
                        'Cast iron and teak',
                        'Stainless steel',
                        'Brass and polymer-based enamel',
                        'Steel, nylon, and 12W fan',
                        'Wood and stainless steel',
                        'Chrome-plated steel and leather',
                        'Enameled steel and plastic',
                        'Silver and teak',
                        'Chrome-plated steel frame, wool upholstery, and painted wood arms',
                        'Steel blade, nylon handle, brass fittings, and leather sheath',
                        'Plastic and stainless steel'
                        ],
                'Paper': [
                        'Offset lithograph',
                        'Pencil on paper',
                        'Lithograph and photolithograph',
                        'Photolithograph',
                        'Lithograph',
                        'Paper twine'
                        ],
                'Plastic': [
                        'Reworked plastic',
                        'Polyethylene',
                        'Recyclable polyester fiber',
                        'Plastic and walnut',
                        'Acrylic'
                        ],
                'Other': [
                        'Satin damask',
                        'Video game software'
                        ]
                }


def group_mediums(x):
    for group, names in material_map.items():
        if x in names:
            return group
    return x


moma.Medium.apply(group_mediums).value_counts()

Medium
Glass       63
Wood        28
Ceramics    20
Metal       11
Paper        7
Plastic      6
Other        2
Name: count, dtype: int64

A copy adjusted for visualisation tool constraints

In [143]:
moma_upd_medium = moma.copy()
moma_upd_medium.Medium = moma_upd_medium.Medium.apply(group_mediums)
moma_upd_medium.to_csv('moma_upd_medium.csv')

### Finna

In [352]:
finna

,id,images,nonPresenterAuthors,subjects,title,year,IndividualAuthor
0,lappeenrannanmuseot.0188C337-B8BA-4027-B85E-AE...,['/Cover/Show?source=Solr&id=lappeenrannanmuse...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[['huonekalut'], ['naulakot'], ['seinähyllyt']...",naulakko; seinänaulakko,1930.0,Alvar Aalto
1,lappeenrannanmuseot.3db2044e-1b19-4514-903d-5d...,['/Cover/Show?source=Solr&id=lappeenrannanmuse...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[['tuolit'], ['sairaalat'], ['tuberkuloosipara...",tuoli,1930.0,Alvar Aalto
2,lappeenrannanmuseot.48FD4DBA-A6F7-4184-AB74-77...,['/Cover/Show?source=Solr&id=lappeenrannanmuse...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[['muotoilu'], ['huonekalut'], ['päiväkodit'],...",selkänojallinen pikkutuoli,1930.0,Alvar Aalto
3,lappeenrannanmuseot.6219dd3c-87f5-470e-bd59-04...,['/Cover/Show?source=Solr&id=lappeenrannanmuse...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[['tuolit'], ['sairaalat'], ['tuberkuloosipara...",tuoli,1930.0,Alvar Aalto
4,lappeenrannanmuseot.67B47DF2-AA24-4415-A480-30...,['/Cover/Show?source=Solr&id=lappeenrannanmuse...,"[{'name': 'Aalto, Alvar', 'role': 'suunnitteli...","[['naulakot'], ['huonekalut'], ['seinähyllyt']]",naulakko; seinänaulakko,1930.0,Alvar Aalto
...,...,...,...,...,...,...,...
1659,postimuseo.FE49BC51-EB37-4BC7-B019-820136B04F71,['/Cover/Show?source=Solr&id=postimuseo.FE49BC...,"[{'name': 'Saarinen, Eliel', 'role': 'tekijä',...","[['luonnokset'], ['postimerkkien valmistus'], ...","vedos; postimerkin mallivedos, ns. Bernin arkk...",NaN,Eliel Saarinen
1660,srm.166901944222200,['/Cover/Show?source=Solr&id=srm.1669019442222...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[['huonekalut'], ['istuimet'], ['toimistokalus...",istuin; tuoli,NaN,Eliel Saarinen
1661,srm.166903323327000,['/Cover/Show?source=Solr&id=srm.1669033233270...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[['huonekalut'], ['istuimet'], ['tuolit']]",istuin; tuoli,NaN,Eliel Saarinen
1662,srm.166903329663700,['/Cover/Show?source=Solr&id=srm.1669033296637...,"[{'name': 'Eliel Saarinen', 'role': 'suunnitte...","[['asemaravintolat'], ['huonekalut'], ['istuim...",asemaravintolan tuoli; istuin; tuoli,NaN,Eliel Saarinen


Subjects assigned to each artwork in Finna's original metadata are too extensive and inconsistent for using them for any kind of classification or categorisation.

In [353]:
finna_subjects = set()

for entry in finna.subjects.unique():
    subj_list = ast.literal_eval(entry)
    for subj in subj_list:
        finna_subjects.add(subj[0])

len(finna_subjects)

1149

Finna's data does not provide sufficient information on materials of artworks. Therefore, to conduct the analysis of mediums identical to the one of MoMA's collection, we need to add this metadata to images explicitly. One way to do it is to perform zero-shot classification using the CLIP model.

*The classification is performend in a separate .ipynb notebook.*

See [this file](Clip_Finna.ipynb).

### MOMA + Finna

In [4]:
moma_full = pd.read_csv("moma_upd_medium.csv", index_col=0)
finna_full = pd.read_csv("finna_upd_medium.csv", index_col=0)

In [5]:
finna_full['Source'] = 'Finna'
finna_full = finna_full[['Source', 'id', 'title', 'year', 'Medium', 'IndividualAuthor', 'ArtistOrder']]
finna_full.rename(columns={'id': 'ObjectID', 'title': 'Title', 'IndividualAuthor': 'Artist', 'year': 'Year'}, inplace=True)

The titles in Finnish from Finna are translated to English for uniformity.

In [6]:
batch_size = 32
finnish_titles = finna_full.Title.values
translations = []

for i in range(0, len(finnish_titles), batch_size):
    i_end = i + batch_size if i + batch_size < len(finnish_titles) else len(finnish_titles)
    batch_sentences = "! ".join([t.capitalize() for t in finnish_titles[i:i_end]])
    batch_results = ts.translate_text(batch_sentences, from_language='fi', to_language='en', translator='google')
    translations.extend(['; '.join([w.capitalize() for w in result.split('; ')]) for result in batch_results.split('! ')])

finna_full.Title = translations

In [11]:
moma_full['Source'] = 'MoMA'
moma_full = moma_full[['Source', 'ObjectID', 'Title', 'ApproxDate', 'Medium', 'Artist', 'ArtistOrder']]
moma_full.rename(columns={'ApproxDate': 'Year'}, inplace=True)

artists.rename(columns={'DisplayName': 'Artist'}, inplace=True)

Merging two datasets into one

In [12]:
all_works = pd.concat([moma_full, finna_full]).merge(artists.drop('ArtistBio', axis=1))
all_works

,Source,ObjectID,Title,Year,Medium,Artist,ArtistOrder,ConstituentID,Nationality,Gender,BeginDate,EndDate,Wiki QID,ULAN
0,MoMA,1091,Prisma Tumblers,1967.0,Glass,Kaj Franck,8,1968,Finnish,male,1911,1989,Q909809,500103402.0
1,MoMA,1184,Tea Strainer,1955.0,Metal,Bertel Gardberg,10,7381,Finnish,male,1916,2007,NaN,NaN
2,MoMA,1185,Triennale Flatware,1957.0,Metal,Bertel Gardberg,10,7381,Finnish,male,1916,2007,NaN,NaN
3,MoMA,1572,Kilta Storage Container,1948.0,Ceramics,Kaj Franck,8,1968,Finnish,male,1911,1989,Q909809,500103402.0
4,MoMA,1583,Kilta Storage Container,1948.0,Ceramics,Kaj Franck,8,1968,Finnish,male,1911,1989,Q909809,500103402.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1788,Finna,postimuseo.FE49BC51-EB37-4BC7-B019-820136B04F71,"Print; Model print of a postage stamp, so-call...",NaN,Paper,Eliel Saarinen,27,64766,Finnish,male,1873,1950,NaN,NaN
1789,Finna,srm.166901944222200,Seat; Chair,NaN,Wood,Eliel Saarinen,27,64766,Finnish,male,1873,1950,NaN,NaN
1790,Finna,srm.166903323327000,Seat; Chair,NaN,Wood,Eliel Saarinen,27,64766,Finnish,male,1873,1950,NaN,NaN
1791,Finna,srm.166903329663700,Station restaurant chair; Seat; Chair,NaN,Wood,Eliel Saarinen,27,64766,Finnish,male,1873,1950,NaN,NaN


Final update of years. For artworks that do not have a date attribution and other artworks by the same author to refer to for a median year value, I roughly estimate it as a middle of an artist's life.

In [14]:
years_upd = []
for row in all_works.itertuples():
    if np.isnan(row.Year):
        author = row.Artist
        subset = all_works[all_works.Artist == author].Year
        median_year = subset.median()
        if np.isnan(median_year):
            year = row.BeginDate + round((row.EndDate - row.BeginDate) / 2)
        else:
            year = round(median_year)
    else:
        year = row.Year
    years_upd.append(year)

all_works.Year = years_upd

In [15]:
all_works.to_csv("moma_vs_finna_full.csv")

In [16]:
all_works.query("Source == 'MoMA'").to_csv("moma_full_with_artists.csv")
all_works.query("Source == 'Finna'").to_csv("finna_full_with_artists.csv")

In [17]:
all_works.Medium.value_counts()

Medium
Glass       443
Ceramics    424
Other       277
Paper       221
Wood        208
Metal       106
Textile      91
Plastic      23
Name: count, dtype: int64

Subset with reduced number of mediums suitable for visualisation in PIDA – losing just above 100 works

In [19]:
main_materials = ['Glass', 'Ceramics', 'Paper', 'Wood', 'Metal', 'Other']
all_works_reduced = all_works[all_works.Medium.isin(['Ceramics', 'Glass', 'Paper', 'Wood', 'Metal', 'Other'])]
all_works_reduced['MaterialOrder'] = all_works_reduced.Medium.apply(lambda x: main_materials.index(x) + 1)
all_works_reduced.to_csv("moma_vs_finna_reduced_medium.csv")

## Visualisations

In [207]:
def get_plot(chart_type, title, data, range_x=None, range_y=None,
              x=None, y=None, group=None, category_orders=None, text=None,
              nbins=None, orient=None, barmode='relative', extra_hist=None,
              x_name=None, y_name=None, hovertemplate=None, hover_titles=[],
              color_list=None, color_dict=None, color_scale=None, template="plotly_white"):

  # Base charts
  if chart_type == 'scatter':
    fig = px.scatter(data, x=x, y=y, color=group, size=[10] * data.shape[0],
                      range_x=range_x, range_y=range_y,
                      color_discrete_sequence=color_list,
                      color_discrete_map=color_dict,
                      template=template, title=title)

  if chart_type == 'bar':
    fig = px.bar(data, x=x, y=y, color=group, text=text,
                orientation=orient, barmode=barmode,
                category_orders=category_orders,
                color_discrete_sequence=color_list,
                color_discrete_map=color_dict,
                color_continuous_scale=color_scale,
                template=template,
                title=title)

  if chart_type == 'hist':
    fig = px.histogram(data, x=x, y=y, color=group, nbins=nbins,
                       barmode=barmode, histnorm='probability',
                       opacity=0.7, color_discrete_sequence=color_list,
                       color_discrete_map=color_dict, marginal=extra_hist,
                       template=template, title=title)
    
  if chart_type == 'box':
    fig = px.box(data, x=x, y=y, color=group,
                 category_orders=category_orders,
                 color_discrete_sequence=color_list,
                 color_discrete_map=color_dict,
                 template=template,
                 title=title)
  
  if chart_type == 'treemap':
    fig = px.treemap(data, path=x, values=y, color=group,
                 color_discrete_sequence=color_list,
                 color_discrete_map=color_dict,
                 template=template,
                 title=title)
    fig.data[0].textinfo = 'label+value'
    fig.data[0].texttemplate='<b>%{label}</b><br>%{value:.2r}'

  # Updates
  fig.update_xaxes(title=x_name)
  fig.update_yaxes(title=y_name)

  fig.update_traces(hovertemplate=hovertemplate)
  if not hovertemplate:
    fig.update_traces(hoverinfo='skip')

  for id, val in enumerate(hover_titles):
    fig.data[id].name = ""
    fig.data[id].hovertemplate = val + '<br>' + fig.data[id].hovertemplate

  fig.update_layout(height=800, width=1200, font_family="Futura", title_font_size=24,
                    showlegend=False, coloraxis_showscale=False,
                    plot_bgcolor="#FFFFFF", paper_bgcolor="#FFFFFF")
  return fig

In [145]:
colors_of_materials = {'Wood': '#ff913d', 
                       'Plastic': '#E74B4B', 
                       'Metal': '#A6DEE4', 
                       'Paper': '#EDCA6B', 
                       'Ceramics': '#DC95B7', 
                       'Other': "#BEBFB3", 
                       'Glass': '#929ADD', 
                       'Textile':'#33B071'}

colors_of_sources = {'MoMA': "#E39D48", 
                     'Finna': "#685FA7"}

colors_of_genders = {'male': "#3F6519", 
                     'female': "#A75F96"}

### Artists

In [227]:
artist_shares = all_works.groupby('Source').Artist.value_counts(normalize=True).reset_index().sort_values('proportion', ascending=False)

plot = get_plot(chart_type='bar', title=f"<br>Designers' representation in <span style='color:{colors_of_sources['MoMA']}'><b>MoMA's</b></span> and <span style='color:{colors_of_sources['Finna']}'><b>Finna's</b></span> collections",
                data=artist_shares, x='Artist', y='proportion', group="Source", barmode='group',
                x_name="", y_name="Proportion",
                hovertemplate="%{x}<br>%{y}",
                color_dict=colors_of_sources)

plot.show()

### Materials

In [ ]:
material_shares = all_works.groupby('Source').Medium.value_counts(normalize=True).reset_index().sort_values('proportion', ascending=False)

plot = get_plot(chart_type='bar', title=f"<br>Material composition of <span style='color:{colors_of_sources['MoMA']}'><b>MoMA's</b></span> and <span style='color:{colors_of_sources['Finna']}'><b>Finna's</b></span> collections",
                data=material_shares, x='Medium', y='proportion', group="Source", barmode='group',
                x_name="", y_name="Proportion",
                hovertemplate="%{x}<br>%{y}",
                color_dict=colors_of_sources)

plot.show()

In [103]:
plot = get_plot(chart_type='treemap', title="<br>Materials of <b>MoMA</b>", group='Medium',
                data=material_shares.query("Source == 'MoMA'"), x=['Medium'], y='proportion',
                color_dict=colors_of_materials)

plot.show()

In [104]:
plot = get_plot(chart_type='treemap', title="<br>Materials at <b>Finna</b>", group='Medium',
                data=material_shares.query("Source == 'Finna'"), x=['Medium'], y='proportion',
                color_dict=colors_of_materials)

plot.show()

In [ ]:
plot = get_plot(chart_type='scatter', title=f"<br>Temporal distribution of materials in <b>MoMA's</b> collection", range_x=[1860, 2020],
                data=all_works.query("Source == 'MoMA'").sort_values('Medium'), x='Year', y='Medium', group="Medium",
                color_dict=colors_of_materials)

plot.show()

In [205]:
plot = get_plot(chart_type='scatter', title=f"<br>Temporal distribution of materials in <b>Finna's</b> collection",
                data=all_works.query("Source == 'Finna'").sort_values('Medium'), x='Year', y='Medium', group="Medium",
                color_dict=colors_of_materials)

plot.show()

In [208]:
plot = get_plot(chart_type='box', title=f"<br>Temporal distribution of materials in <b>MoMA's</b> collection", range_x=[1860, 2020],
                data=all_works.query("Source == 'MoMA'").sort_values('Medium'), x='Year', y='Medium', group="Medium",
                color_dict=colors_of_materials)

plot.show()

In [209]:
plot = get_plot(chart_type='box', title=f"<br>Temporal distribution of materials in <b>Finna's</b> collection",
                data=all_works.query("Source == 'Finna'").sort_values('Medium'), x='Year', y='Medium', group="Medium",
                color_dict=colors_of_materials)

plot.show()

### Creation Dates

In [200]:
year_shares = all_works.groupby('Source').Year.value_counts(normalize=True).reset_index().sort_values('proportion', ascending=False)

plot = get_plot(chart_type='hist', title=f"<br>Temporal distribution of works in <span style='color:{colors_of_sources['MoMA']}'><b>MoMA's</b></span> and <span style='color:{colors_of_sources['Finna']}'><b>Finna's</b></span> collections – Years",
                data=all_works, x='Year', group="Source", nbins=len(all_works.Year.unique())*2, # int(np.ceil((all_works.Year.max() - all_works.Year.min()) / 5))
                x_name="", y_name="Proportion", barmode="overlay", extra_hist="box",
                hovertemplate="%{x}<br>%{y}",
                color_dict=colors_of_sources)

plot.show()

In [ ]:
year_shares = all_works.groupby('Source').Year.value_counts(normalize=True).reset_index().sort_values('proportion', ascending=False)

plot = get_plot(chart_type='hist', title=f"<br>Temporal distribution of works in <span style='color:{colors_of_sources['MoMA']}'><b>MoMA's</b></span> and <span style='color:{colors_of_sources['Finna']}'><b>Finna's</b></span> collections – Decades",
                data=all_works, x='Year', group="Source", nbins=int(np.ceil((all_works.Year.max() - all_works.Year.min()) / 10)),
                x_name="", y_name="Proportion", barmode="group",
                hovertemplate="%{x}<br>%{y}",
                color_dict=colors_of_sources)

plot.update_layout(xaxis = dict(
                    tickmode='array',
                    tickvals = list(range(1860, 2020, 10)),
                    ticktext = list(range(1860, 2020, 10)),
                    ))

plot.show()